# TPOSE24 validation vs OISST (sea-surface temperature)

Daily NOAA OISST is compared to the model surface temperature (`THETA` at the topmost level) over the model domain and run period (Oct–Dec 2012).

The model is daily-averaged and regridded onto the (coarser) 0.25° OISST grid; all spatial statistics use cos(latitude) weighting.  Supply any number of runs in `MODELS` — the figures scale automatically.

**1-D diagnostics**: domain-mean RMSE & bias time series with a box-plot summary; time-mean SST and bias as a function of longitude (equatorial band) and of latitude (zonal mean).

**2-D diagnostics**: maps of time-mean SST, of the SST bias (model − OISST), and of the RMSE for each run.

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  MODEL RUNS TO VALIDATE
#  Each entry is (label_for_figures, run_directory).
#  Add or remove entries here — every figure below updates its number
#  of lines / panels automatically.
# ════════════════════════════════════════════════════════════════════
MODELS = [
    ('Ri7', '/data/SO3/edavenport/tpose24/oct2012_3month_transp_cons'),
    ('Ri3', '/data/SO3/edavenport/tpose24/oct2012_TP6Vel_3month_Ri3'),
    ('Ri5', '/data/SO3/edavenport/tpose24/oct2012_TP6Vel_3month_Ri5'),
]

OUTDIR    = 'OISST_comparison'
OISST_OBS = '/data/SO3/edavenport/tpose6/oisst_data/oisst_equatorial_pacific_2012to2013.nc'


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cmocean.cm as cmo

import obs_validation_utils as u

plt.rcParams.update({'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11})
os.makedirs(OUTDIR, exist_ok=True)
print(f'{len(MODELS)} model run(s):', [m[0] for m in MODELS])
print('Figures ->', OUTDIR)


In [ ]:
# ── Load OISST once (longitude -> 0:360) ─────────────────────────────
oisst_ds = u.to_0360(xr.open_dataset(OISST_OBS))
print('OISST time:', str(oisst_ds.time.values[0])[:10], '->',
      str(oisst_ds.time.values[-1])[:10])


In [ ]:
# ── Load every model, regrid onto OISST, align in time ───────────────
# All runs share the same grid, so the OISST domain subset is built once
# from the first run and reused.
results = {}      # label -> dict of arrays
obs_sst = None    # OISST aligned to the (shared) model days

for i, (label, run_dir) in enumerate(MODELS):
    print(f'[{label}] loading {run_dir}')
    ds   = u.load_tpose24(run_dir, prefix=['diag_state'])
    sst  = ds.THETA.isel(Z=0)
    sst  = sst.where(sst != 0)            # mask land
    sstd = u.daily_mean(sst)

    if obs_sst is None:
        dom = u.model_domain(ds)
        print('   domain (lon0,lon1,lat0,lat1) =',
              tuple(round(d, 2) for d in dom))
        t0 = str(sstd.time.values[0])[:10]
        t1 = str(sstd.time.values[-1])[:10]
        obs_sub = (u.subset_domain(oisst_ds.sst, *dom)
                     .sel(time=slice(t0, t1)).compute())

    mod = u.regrid_model_to_obs(sstd, obs_sub.longitude,
                                obs_sub.latitude).compute()
    mod_a, obs_a = u.align_daily(mod, obs_sub)
    if obs_sst is None:
        obs_sst = obs_a
        lat = obs_sst.latitude.values
        lon = obs_sst.longitude.values
        time = obs_sst.time.values

    m = mod_a.values
    o = obs_a.values
    results[label] = dict(
        field    = mod_a,
        rmse_t   = u.weighted_spatial_rmse(m, o, lat),
        bias_t   = u.weighted_spatial_bias(m, o, lat),
        mean_map = np.nanmean(m, axis=0),
        bias_map = np.nanmean(m - o, axis=0),
        rmse_map = np.sqrt(np.nanmean((m - o) ** 2, axis=0)),
    )
    print(f'   RMSE {np.nanmean(results[label]["rmse_t"]):.3f} degC | '
          f'bias {np.nanmean(results[label]["bias_t"]):.3f} degC')

obs_mean = np.nanmean(obs_sst.values, axis=0)
days = np.arange(len(time))
print('Aligned days:', len(time))


In [ ]:
# ── 1-D: domain-mean RMSE & bias time series + box-plot summary ──────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5),
                         gridspec_kw={'width_ratios': [3, 3, 1.4]})
tt = pd.to_datetime(time)

for i, (label, _) in enumerate(MODELS):
    c = u.model_color(i)
    axes[0].plot(tt, results[label]['rmse_t'], color=c, lw=1.8, label=label)
    axes[1].plot(tt, results[label]['bias_t'], color=c, lw=1.8, label=label)

axes[0].set_title('SST RMSE vs OISST'); axes[0].set_ylabel('RMSE (°C)')
axes[1].axhline(0, color='k', lw=0.7, ls=':')
axes[1].set_title('SST bias (model − OISST)'); axes[1].set_ylabel('Bias (°C)')
for ax in axes[:2]:
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

box = [results[l]['rmse_t'] for l, _ in MODELS]
bp = axes[2].boxplot(box, labels=[l for l, _ in MODELS], patch_artist=True)
for k, patch in enumerate(bp['boxes']):
    patch.set_facecolor(u.model_color(k)); patch.set_alpha(0.6)
axes[2].set_title('RMSE distribution'); axes[2].set_ylabel('RMSE (°C)')
axes[2].grid(alpha=0.3, axis='y')
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=45, ha='right')

fig.tight_layout()
fig.savefig(f'{OUTDIR}/oisst_1d_rmse_bias_timeseries.png', dpi=150,
            bbox_inches='tight')
plt.show()


In [ ]:
# ── 1-D: time-mean SST & bias vs longitude (equatorial band |lat|<2°) ─
band = np.abs(lat) <= 2.0
w = np.cos(np.deg2rad(lat[band]))[:, None]

def band_mean(arr2d):
    a = arr2d[band, :]
    return np.nansum(a * w, axis=0) / np.nansum(w * np.isfinite(a), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(lon, band_mean(obs_mean), color='k', lw=2.5, label='OISST')
for i, (label, _) in enumerate(MODELS):
    axes[0].plot(lon, band_mean(results[label]['mean_map']),
                 color=u.model_color(i), lw=1.8, label=label)
    axes[1].plot(lon, band_mean(results[label]['bias_map']),
                 color=u.model_color(i), lw=1.8, label=label)

axes[0].set_title('Time-mean SST, 2°S–2°N'); axes[0].set_ylabel('SST (°C)')
axes[1].axhline(0, color='k', lw=0.7, ls=':')
axes[1].set_title('SST bias (model − OISST), 2°S–2°N')
axes[1].set_ylabel('Bias (°C)')
for ax in axes:
    ax.set_xlabel('Longitude (°E)'); ax.grid(alpha=0.3); ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(f'{OUTDIR}/oisst_1d_zonal_structure.png', dpi=150,
            bbox_inches='tight')
plt.show()


In [ ]:
# ── 1-D: time-mean SST & bias vs latitude (zonal mean) ───────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
axes[0].plot(np.nanmean(obs_mean, axis=1), lat, color='k', lw=2.5,
             label='OISST')
for i, (label, _) in enumerate(MODELS):
    axes[0].plot(np.nanmean(results[label]['mean_map'], axis=1), lat,
                 color=u.model_color(i), lw=1.8, label=label)
    axes[1].plot(np.nanmean(results[label]['bias_map'], axis=1), lat,
                 color=u.model_color(i), lw=1.8, label=label)

axes[0].set_title('Zonal-mean SST'); axes[0].set_xlabel('SST (°C)')
axes[0].set_ylabel('Latitude (°N)')
axes[1].axvline(0, color='k', lw=0.7, ls=':')
axes[1].set_title('Zonal-mean bias (model − OISST)')
axes[1].set_xlabel('Bias (°C)')
for ax in axes:
    ax.axhline(0, color='gray', lw=0.6, ls=':'); ax.grid(alpha=0.3)
    ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(f'{OUTDIR}/oisst_1d_meridional_structure.png', dpi=150,
            bbox_inches='tight')
plt.show()


In [ ]:
# ── 2-D: time-mean SST maps (OISST + each model) ─────────────────────
n = len(MODELS) + 1
fig, axes = plt.subplots(1, n, figsize=(4.6 * n, 4), sharex=True,
                         sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)
vmin = np.floor(np.nanmin(obs_mean)); vmax = np.ceil(np.nanmax(obs_mean))

im = axes[0].pcolormesh(lon, lat, obs_mean, cmap=cmo.thermal,
                        vmin=vmin, vmax=vmax, shading='auto')
axes[0].set_title('OISST')
for i, (label, _) in enumerate(MODELS):
    axes[i + 1].pcolormesh(lon, lat, results[label]['mean_map'],
                           cmap=cmo.thermal, vmin=vmin, vmax=vmax,
                           shading='auto')
    axes[i + 1].set_title(label)
for ax in axes:
    ax.axhline(0, color='gray', lw=0.5, ls=':'); ax.set_xlabel('Lon (°E)')
axes[0].set_ylabel('Lat (°N)')
fig.colorbar(im, ax=axes, label='Time-mean SST (°C)', shrink=0.85)
fig.suptitle('Time-mean SST', y=1.04)
fig.savefig(f'{OUTDIR}/oisst_2d_mean_sst.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 2-D: SST bias maps (model − OISST) ───────────────────────────────
n = len(MODELS)
bmax = np.nanmax([np.nanpercentile(np.abs(results[l]['bias_map']), 99)
                  for l, _ in MODELS])
fig, axes = plt.subplots(1, n, figsize=(4.6 * n, 4), sharex=True,
                         sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)
for i, (label, _) in enumerate(MODELS):
    im = axes[i].pcolormesh(lon, lat, results[label]['bias_map'],
                            cmap=cmo.balance, vmin=-bmax, vmax=bmax,
                            shading='auto')
    axes[i].set_title(f'{label} − OISST')
    axes[i].axhline(0, color='gray', lw=0.5, ls=':')
    axes[i].set_xlabel('Lon (°E)')
axes[0].set_ylabel('Lat (°N)')
fig.colorbar(im, ax=axes, label='SST bias (°C)', shrink=0.85)
fig.suptitle('Time-mean SST bias (model − OISST)', y=1.04)
fig.savefig(f'{OUTDIR}/oisst_2d_bias_maps.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 2-D: SST RMSE maps ───────────────────────────────────────────────
n = len(MODELS)
rmax = np.nanmax([np.nanpercentile(results[l]['rmse_map'], 99)
                  for l, _ in MODELS])
fig, axes = plt.subplots(1, n, figsize=(4.6 * n, 4), sharex=True,
                         sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)
for i, (label, _) in enumerate(MODELS):
    im = axes[i].pcolormesh(lon, lat, results[label]['rmse_map'],
                            cmap=cmo.amp, vmin=0, vmax=rmax, shading='auto')
    axes[i].set_title(label)
    axes[i].axhline(0, color='gray', lw=0.5, ls=':')
    axes[i].set_xlabel('Lon (°E)')
axes[0].set_ylabel('Lat (°N)')
fig.colorbar(im, ax=axes, label='SST RMSE (°C)', shrink=0.85)
fig.suptitle('SST RMSE vs OISST', y=1.04)
fig.savefig(f'{OUTDIR}/oisst_2d_rmse_maps.png', dpi=150, bbox_inches='tight')
plt.show()
